API JSON Parsing

In [12]:
import httpx
from pathlib import Path
import json


In [22]:
URL_VULNERABILITY = 'https://cveawg.mitre.org/api/cve/CVE-2021-44228'
FILE_VULNERABILITY = Path().parent / 'CVE.json'
print(FILE_VULNERABILITY.absolute())

e:\Projects\FEFU_SD\2026-03-13\CVE.json


In [8]:
with httpx.Client() as client:
    result = client.get(URL_VULNERABILITY)


In [20]:
print(result)
print(result.text[:50])
print(type(result.text))
print(result.content[:50])
print(type(result.content))
new_result = result.json()
print(type(new_result))

<Response [200 OK]>
{"dataType":"CVE_RECORD","dataVersion":"5.1","cveM
<class 'str'>
b'{"dataType":"CVE_RECORD","dataVersion":"5.1","cveM'
<class 'bytes'>
<class 'dict'>


In [23]:
with FILE_VULNERABILITY.open('w') as f:
    json.dump(new_result, f, indent=2)

In [25]:
cve_list = ('CVE-2021-44228','CVE-2022-30190')
with httpx.Client() as client:
    for cve in cve_list:
        url = f'https://cveawg.mitre.org/api/cve/{cve}'
        result = client.get(url)
        with (Path().parent / f'{cve}.json').open('w') as f:
            json.dump(result.json(), f, indent=2)

RSS parsing

In [26]:
URL_RSS = 'https://api.msrc.microsoft.com/update-guide/rss'

In [27]:
with httpx.Client() as client:
    rss = client.get(URL_RSS)


In [31]:
rss.text[:50]
RSS_FILE = Path().parent / 'rss.xml'
with RSS_FILE.open('wb') as f:
    f.write(rss.content)

In [32]:
import xml.etree.ElementTree as ET
tree = ET.parse(RSS_FILE)
root = tree.getroot()

In [36]:
print(root.tag)
print(root.attrib)
print(root.attrib.get('version'))
print(root.attrib.get('version1', 'none'))


rss
{'version': '2.0'}
2.0
none


In [44]:
channel = root.find('channel')
title = channel.find('title')
if title is not None:
    print(title.text)

MSRC Security Update Guide


In [45]:
item_list = channel.findall('item')
print(len(item_list))

3230


Parsing sites

In [50]:
FEFU_EDUCATION_URL = 'https://www.dvfu.ru/education/'


In [54]:
with httpx.Client() as client:
    fefu_result = client.get(FEFU_EDUCATION_URL)
with (Path().parent / 'fefu_education.html').open('wb') as f:
    f.write(fefu_result.content)


Using BeautifulSoup4

In [71]:
from bs4 import BeautifulSoup
from pprint import pprint
from urllib.parse import urljoin


In [122]:
page = BeautifulSoup(fefu_result.text)

def parse_by_bs4(bs_item: BeautifulSoup) -> dict[str, list[dict[str, str]]]:
    titles = bs_item.find_all('h4',attrs={'class':'title'})
    result = {}
    for title in titles:
        title_text = title.text.strip()
        div_tag = title.find_parent('div', attrs={'class':'__block'})
        if div_tag is None:
            print(f'Нет тэга div для заголовка {title_text}')
            continue

        ul_tag = div_tag.find_next('ul', attrs={'class':'dash-list'})
        links = ul_tag.find_all('a')
        result[title_text] = list()
        for link in links:
            url = link.attrs['href']
            url_title = link.text.strip()
            result[title_text].append({
                'url': urljoin(FEFU_EDUCATION_URL, url),
                'url_text': url_title,
            })
    return result

fefu_education_blocks = parse_by_bs4(page)


In [90]:
for title, url_list in fefu_education_blocks.items():
    print(f'"{title}" содержит {len(url_list)} ссылок')

"Учебный процесс" содержит 10 ссылок
"Студенческие сервисы" содержит 12 ссылок
"Развитие талантов" содержит 5 ссылок
"Информация об обучении" содержит 5 ссылок
"Дополнительное образование" содержит 4 ссылок
"Довузовское образование" содержит 3 ссылок


In [123]:
with(Path().parent / 'fefu_bs4.json').open('w', encoding='utf-8') as f:
    json.dump(fefu_education_blocks, f, indent=2, ensure_ascii=False)

Using XPath (lxml)

In [81]:
from lxml.html import fromstring
from collections import defaultdict

In [119]:
def parse_by_xpath(lxml_item) -> dict[str, list[dict[str, str]]]:
    result = defaultdict(list)
    title_xpath = '//h4[@class="title"]'
    links_xpath = './parent::div/following-sibling::ul[@class="dash-list"]/li/a'

    for title_tag in lxml_item.xpath(title_xpath):
        title_text = title_tag.text.strip()
        for link_tag in title_tag.xpath(links_xpath):
            url = link_tag.get('href')
            url_title = link_tag.text.strip()
            result[title_text].append({
                'url': urljoin(FEFU_EDUCATION_URL, url),
                'url_text': url_title,
            })
    return result


page = fromstring(fefu_result.text)
fefu_education_blocks = parse_by_xpath(page)

In [112]:
for title, url_list in fefu_education_blocks.items():
    print(f'"{title}" содержит {len(url_list)} ссылок')

"Учебный процесс" содержит 10 ссылок
"Студенческие сервисы" содержит 12 ссылок
"Развитие талантов" содержит 5 ссылок
"Информация об обучении" содержит 5 ссылок
"Дополнительное образование" содержит 4 ссылок
"Довузовское образование" содержит 3 ссылок


In [121]:
with(Path().parent / 'fefu_xpath.json').open('w', encoding='utf-8') as f:
    json.dump(fefu_education_blocks, f, indent=2, ensure_ascii=False)